# Chatbot

This notebook turns the local `../scripts/llm_client` chatbot into a Jupyter chat panel. It can run offline without robot tools, or bind the LLM tools to `sdk_client.Robot` and optionally speak replies on the robot speaker.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import the same chat and robot-tool functions used by the CLI.


In [2]:
import os
from types import SimpleNamespace

import ipywidgets as widgets
from IPython.display import display

from llm_client import chat as chat_module
from llm_client.chat import send_chat_with_tool_usage_loop
from llm_client.cli import DEFAULT_SYSTEM, PROVIDERS, say_safely
from llm_client.robot_tools import build_robot_tools
from sdk_client import Robot


Configure the provider and build the chat state. Set `use_robot_tools` to true only when the robot is in a safe state for tool calls.


In [3]:
provider_name = os.environ.get("G1_CHAT_PROVIDER", "anthropic")
provider = PROVIDERS[provider_name]
model = os.environ.get("G1_CHAT_MODEL", provider["model"])
base = os.environ.get("G1_CHAT_BASE", provider["base"])
api_key = os.environ.get(provider["env_var"])
if api_key:
    chat_module.dnabot_auth = SimpleNamespace(get_auth_header=lambda: {"Authorization": f"Bearer {api_key}"})
else:
    print(f"Set {provider['env_var']} before sending messages.")

use_robot_tools = False
speak_replies = False
robot = None
tools = {}
schemas = []
messages = [{"role": "system", "content": DEFAULT_SYSTEM}]
print(f"provider={provider_name} model={model} base={base}")


Set ANTHROPIC_API_KEY before sending messages.
provider=anthropic model=claude-sonnet-4-6 base=https://api.anthropic.com/v1


Optional robot binding. Run this cell after setting `use_robot_tools = True` if the robot should expose move/reach/grab/release tools to the model.


In [4]:
if use_robot_tools:
    robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, safety_boot=False, auto_start_sensors=False)
    tools, schemas = build_robot_tools(robot)
    print("Robot tools enabled:", list(tools))
else:
    print("Robot tools disabled. Set use_robot_tools=True and rerun this cell to enable them.")


Robot tools disabled. Set use_robot_tools=True and rerun this cell to enable them.


Run the chat panel. Tool calls are displayed in the log; replies can be spoken when `speak_replies` is enabled.


In [5]:
prompt = widgets.Textarea(placeholder="Type a message...", layout=widgets.Layout(width="100%", height="90px"))
send = widgets.Button(description="Send", button_style="success")
clear = widgets.Button(description="Clear")
speak = widgets.Checkbox(value=speak_replies, description="speak replies")
chat_log = widgets.Textarea(layout=widgets.Layout(width="100%", height="420px"), disabled=True)


def add(line):
    chat_log.value = (chat_log.value + line + "\n")[-12000:]


def tool_printer(name, args, output):
    add(f"[tool] {name}({args}) -> {output}")


def on_send(_):
    text = prompt.value.strip()
    if not text:
        return
    prompt.value = ""
    messages.append({"role": "user", "content": text})
    add(f"you> {text}")
    try:
        content = send_chat_with_tool_usage_loop(
            model_key=model,
            messages=messages,
            base=base,
            tools=tools,
            tool_schemas=schemas,
            tool_choice="auto" if tools else None,
            max_iterations=8,
            on_tool_call=tool_printer,
        )
        messages.append({"role": "assistant", "content": content})
        add(f"bot> {content}")
        if speak.value:
            say_safely(robot, content)
    except Exception as exc:
        messages.pop()
        add(f"error> {exc}")


def on_clear(_):
    messages[:] = [{"role": "system", "content": DEFAULT_SYSTEM}]
    chat_log.value = ""

send.on_click(on_send)
clear.on_click(on_clear)
display(widgets.VBox([prompt, widgets.HBox([send, clear, speak]), chat_log]))
